# 04 – Full pipeline demo video
In this notebook, we render the final demo video. The pipeline detects
people, tracks them, estimates their pose, classifies their state, and draws
the triage overlay on a held-out Okutama test video. It uses the fine-tuned
weights from notebooks 01 and 03.


In [ ]:
# Install the packages we need.
!pip -q install ultralytics rtmlib onnxruntime-gpu
import torch
print('CUDA available:', torch.cuda.is_available())

# Load the project code.
from pathlib import Path
SRC_ZIP = None
if SRC_ZIP is None:
    from google.colab import files
    up = files.upload()
    SRC_ZIP = next(iter(up))
!mkdir -p /content/project && unzip -q -o "$SRC_ZIP" -d /content/project
import sys
sys.path.insert(0, '/content/project/src')
sys.path.insert(0, '/content/project')

# Results are saved to Google Drive so they survive a disconnect.
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/sar_project_results'); OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Download Okutama-Action files from the public Dropbox folder.
OKUTAMA_BASE = ('https://www.dropbox.com/scl/fo/9qvpsb3fsamvqzsa12149/'
                'APTyV-f01XLnJ0WFpZSBLOE?preview={name}&rlkey=7u7131amaul29amyr4jbnnu03&dl=1')

def fetch_okutama(name, dest='/content/data/okutama'):
    """Download and unpack one Okutama archive unless it is already present."""
    import subprocess, pathlib
    d = pathlib.Path(dest); d.mkdir(parents=True, exist_ok=True)
    zp = d / name
    if not zp.exists():
        subprocess.run(['curl', '-L', '-o', str(zp), OKUTAMA_BASE.format(name=name)], check=True)
    subprocess.run(['unzip', '-q', '-o', str(zp), '-d', str(d)], check=True)
    return d


In [ ]:
fetch_okutama('TestSetVideos.zip')   # Held-out footage the models have never seen.
import glob
test_videos = sorted(glob.glob('/content/data/okutama/**/*.mov', recursive=True))
print(test_videos[:5])


In [ ]:
import config
config.MODELS_DIR = OUT   # Picks up pose_mlp.pt trained in notebook 03.
from render_demo import render
det_weights = str(OUT/'yolo11s_visdrone_best.pt')  # From notebook 01 (or 'yolo11s.pt').
out, n = render(test_videos[0], '/content/demo_raw.mp4', weights=det_weights,
                imgsz=1280, max_frames=1800, device=0, pose_device='cuda',
                caption='EECS 4422 - SAR triage pipeline (held-out footage)')


In [ ]:
# Re-encode with h264 so the video plays everywhere, then keep it in Drive.
!ffmpeg -y -loglevel error -i /content/demo_raw.mp4 -c:v libx264 -pix_fmt yuv420p {OUT}/demo_final.mp4
print('Saved to Drive:', OUT/'demo_final.mp4')
